# 02. Integración multianual de datos GMN

Este cuaderno carga y homogeneiza los archivos de diciembre de 2019 a 2025, aplica los criterios de calidad definidos a partir de la auditoría del notebook pasado y guarda una base integrada para el análisis de las Gemínidas.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq


## 1. Archivos de entrada


In [ ]:
# Localizar la raíz del proyecto y comprobar que estén disponibles los siete archivos de entrada

ruta_actual = Path.cwd().resolve()

raiz_proyecto = next(
    (
        ruta
        for ruta in [ruta_actual, *ruta_actual.parents]
        if (ruta / "pyproject.toml").is_file()
    ),
    None,
)

if raiz_proyecto is None:
    raise FileNotFoundError(
        "No se pudo localizar la raíz del proyecto. "
        "Abre VS Code desde el repositorio de la investigación."
    )

carpeta_raw = raiz_proyecto / "data" / "raw"

archivos_esperados = [
    carpeta_raw / f"traj_summary_monthly_{anio}12.txt"
    for anio in range(2019, 2026)
]

archivos_faltantes = [
    ruta.name
    for ruta in archivos_esperados
    if not ruta.is_file()
]

if archivos_faltantes:
    raise FileNotFoundError(
        "Faltan archivos GMN en data/raw/. "
        "Descarga los archivos indicados en el README antes de ejecutar este cuaderno: "
        + ", ".join(archivos_faltantes)
    )

resumen_archivos = pd.DataFrame(
    {
        "anio": list(range(2019, 2026)),
        "archivo": [ruta.name for ruta in archivos_esperados],
        "tamano_mb": [
            ruta.stat().st_size / 1024 ** 2
            for ruta in archivos_esperados
        ],
    }
)

resumen_archivos["tamano_mb"] = resumen_archivos["tamano_mb"].round(2)

print("Raíz del proyecto:", raiz_proyecto)
print("\nArchivos GMN disponibles:")
display(resumen_archivos)


## 2. Lectura y normalización de los archivos GMN


In [ ]:
# Definir funciones reutilizables para limpiar y reconstruir los encabezados de GMN

def limpiar_texto_encabezado(texto):
    texto_limpio = str(texto).replace("#", "")
    texto_limpio = " ".join(texto_limpio.split())
    
    return texto_limpio

def aplanar_encabezados(columnas_multinivel):
    nombres_aplanados = []
    
    for nombre_principal, nombre_secundario in columnas_multinivel:
        nombre_principal = limpiar_texto_encabezado(nombre_principal)
        nombre_secundario = limpiar_texto_encabezado(nombre_secundario)
        
        if nombre_secundario.startswith("Unnamed"):
            nombre_final = nombre_principal
        else:
            nombre_final = f"{nombre_principal} ({nombre_secundario})"
            
        nombres_aplanados.append(nombre_final)
        
    return nombres_aplanados

def renombrar_columnas_sigma(nombres_columnas):
    nombres_corregidos = nombres_columnas.copy()
    
    for indice, nombre in enumerate(nombres_corregidos):
        if nombre.startswith("+/-"):
            if indice == 0:
                raise ValueError("Se encontró una columna sigma sin una variable anterior.")
            
            variable_anterior = nombres_corregidos[indice - 1]
            nombre_variable = variable_anterior.split()[0]
            
            nombres_corregidos[indice] = f"{nombre_variable}_sigma"
    
    if len(nombres_corregidos) != len(set(nombres_corregidos)):
        nombres_repetidos = sorted({nombre for nombre in nombres_corregidos if nombres_corregidos.count(nombre) > 1})
        
        raise ValueError(f"Se generaron nombres de columnas repetidos: {nombres_repetidos}")
    
    return nombres_corregidos


In [ ]:
# Convertir fechas GMN aceptando explícitamente registros con o sin fracción de segundo

def convertir_fechas_gmn(serie_fechas):
    texto_fechas = serie_fechas.astype("string").str.strip()
    
    fechas_interpretadas = pd.to_datetime(texto_fechas, format = "%Y-%m-%d %H:%M:%S.%f", errors = "coerce", utc = True)
    
    requiere_formato_segundos = fechas_interpretadas.isna() & texto_fechas.notna()
    
    fechas_solo_segundos = pd.to_datetime(texto_fechas.loc[requiere_formato_segundos], format = "%Y-%m-%d %H:%M:%S", errors = "coerce", utc = True)
    
    fechas_interpretadas.loc[requiere_formato_segundos] = fechas_solo_segundos
    
    fechas_invalidas = fechas_interpretadas.isna()
    
    if fechas_invalidas.any():
        ejemplos_invalidos = texto_fechas.loc[fechas_invalidas].head(10).tolist()
        raise ValueError(f"Se encontraron fechas que no coinciden con los formatos GMN aceptados. Ejemplos: {ejemplos_invalidos}")
    
    return fechas_interpretadas


In [ ]:
# Definir una función reutilizable para cargar y preparar estructuralmente un archivo mensual de GMN

def cargar_archivo_gmn(ruta_archivo):
    ruta_archivo = Path(ruta_archivo)
    
    if not ruta_archivo.exists():
        raise FileNotFoundError(f"No se encontró el archivo: {ruta_archivo}")
    
    df_archivo = pd.read_csv(ruta_archivo, engine = "python", sep = ";", skipinitialspace = True, skiprows = [0, 5, 6], header = [0, 1], na_values = ["nan", "None"])
    
    nombres_columnas = aplanar_encabezados(df_archivo.columns)
    nombres_columnas = renombrar_columnas_sigma(nombres_columnas)
    
    df_archivo.columns = (nombres_columnas)
    
    columnas_obligatorias = [
        "Unique trajectory (identifier)",
        "Beginning (UTC Time)",
        "IAU (code)",
        "RAgeo (deg)",
        "RAgeo_sigma",
        "DECgeo (deg)",
        "DECgeo_sigma",
        "Vgeo (km/s)",
        "Vgeo_sigma",
        "Vavg (km/s)",
        "Vavg_sigma",
        "Peak (AbsMag)",
        "Qc (deg)",
        "MedianFitErr (arcsec)",
        "Num (stat)",
        "Beg in (FOV)",
        "End in (FOV)",
        "Participating (stations)",
    ]
    
    columnas_faltantes = [columna for columna in columnas_obligatorias if columna not in df_archivo.columns]
    
    if columnas_faltantes:
        raise ValueError(f"Faltan columnas obligatorias en {ruta_archivo.name}: {columnas_faltantes}")
    
    columnas_texto = ["Unique trajectory (identifier)", "IAU (code)", "Participating (stations)"]
    
    for columna in columnas_texto:
        df_archivo[columna] = df_archivo[columna].astype("string").str.strip().replace("", pd.NA)
        
    columna_fecha = "Beginning (UTC Time)"
    
    df_archivo[columna_fecha] = convertir_fechas_gmn(df_archivo[columna_fecha])
    
    columnas_booleanas = ["Beg in (FOV)", "End in (FOV)"]
    
    for columna in columnas_booleanas:
        if not pd.api.types.is_bool_dtype(df_archivo[columna]):
            valores_booleanos = df_archivo[columna].astype("string").str.strip().str.lower()
            
            valores_invalidos = valores_booleanos.notna() & ~valores_booleanos.isin(["true", "false"])
            
            if valores_invalidos.any():
                ejemplos_invalidos = valores_booleanos.loc[valores_invalidos].drop_duplicates().tolist()
                
                raise ValueError(f"Se encontraron valores booleanos inválidos en {columna}: {ejemplos_invalidos}")
            
            if valores_booleanos.isna().any():
                raise ValueError(f"Se encontraron valores faltantes en {columna}.")
            
            df_archivo[columna] = valores_booleanos.map({"true": True, "false": False}).astype(bool)
            
    codigo_fecha = ruta_archivo.stem.split("_")[-1]
    
    if len(codigo_fecha) != 6 or not codigo_fecha.isdigit():
        raise ValueError(f"No se pudo obtener el año y el mes a partir del nombre: {ruta_archivo.name}")
    
    anio_archivo = int(codigo_fecha[:4])
    mes_archivo = int(codigo_fecha[4:])
    
    if not df_archivo[columna_fecha].dt.year.eq(anio_archivo).all():
        raise ValueError("El año de algunas fechas no coincide con el nombre del archivo.")
    
    if not df_archivo[columna_fecha].dt.month.eq(mes_archivo).all():
        raise ValueError("El mes de algunas fechas no coincide con el nombre del archivo")
    
    df_archivo["source_file"] = ruta_archivo.name
    df_archivo["source_year"] = anio_archivo
    df_archivo["source_month"] = mes_archivo
    
    return df_archivo


## 3. Criterios de calidad


In [ ]:
# Definir una función que añada variables y banderas de calidad trabajando sobre una copia del DataFrame

def agregar_variables_calidad(df_entrada):
    df_calidad = df_entrada.copy(deep = True)
    
    columnas_necesarias = [
        "RAgeo (deg)",
        "RAgeo_sigma",
        "DECgeo (deg)",
        "DECgeo_sigma",
        "Vgeo_sigma",
        "Vavg_sigma",
        "Qc (deg)",
        "MedianFitErr (arcsec)",
        "Num (stat)",
        "Beg in (FOV)",
        "End in (FOV)",
    ]
    
    columnas_faltantes = [columna for columna in columnas_necesarias if columna not in df_calidad.columns]
    
    if columnas_faltantes:
        raise ValueError(f"No se pueden calcular las variables de calidad. Faltan: {columnas_faltantes}")
    
    declinacion_radianes = np.deg2rad(df_calidad["DECgeo (deg)"])
    df_calidad["RAgeo_sigma_cielo"] = df_calidad["RAgeo_sigma"] * np.cos(declinacion_radianes)
    df_calidad["sigma_radiante_aprox"] = np.hypot(df_calidad["RAgeo_sigma_cielo"], df_calidad["DECgeo_sigma"])
    df_calidad["flag_vgeo_sigma_cero"] = df_calidad["Vgeo_sigma"].eq(0)
    df_calidad["flag_vavg_sigma_cero"] = df_calidad["Vavg_sigma"].eq(0)
    
    df_calidad["cumple_qc_15"] = df_calidad["Qc (deg)"].ge(15)
    df_calidad["cumple_error_ajuste_60"] = df_calidad["MedianFitErr (arcsec)"].le(60)
    df_calidad["cumple_vavg_sigma_1"] = df_calidad["Vavg_sigma"].gt(0) & df_calidad["Vavg_sigma"].le(1)
    df_calidad["cumple_sigma_radiante_025"] = df_calidad["sigma_radiante_aprox"].le(0.25)
    
    df_calidad["calidad_radiante_publicada"] = (
        df_calidad["cumple_qc_15"] 
        & df_calidad["cumple_error_ajuste_60"] 
        & df_calidad["cumple_vavg_sigma_1"] 
        & df_calidad["cumple_sigma_radiante_025"]
    )
    
    df_calidad["fov_completo"] = (
        df_calidad["Beg in (FOV)"]
        & df_calidad["End in (FOV)"]
    )
    
    df_calidad["calidad_estandar_provisional"] = (
        df_calidad["calidad_radiante_publicada"]
        & df_calidad["fov_completo"]
    )
    
    df_calidad["calidad_estricta_provisional"] = (
        df_calidad["calidad_estandar_provisional"]
        & df_calidad["Num (stat)"].ge(4)
    )
    
    return df_calidad


In [ ]:
# Preparar las variables de calidad y marcar las trayectorias con información incompleta

def preparar_calidad_gmn(df_entrada):
    df_calidad = agregar_variables_calidad(df_entrada)

    columnas_requeridas_calidad = [
        "RAgeo_sigma",
        "DECgeo_sigma",
        "Vgeo_sigma",
        "Vavg_sigma",
        "MedianFitErr (arcsec)",
    ]

    df_calidad["flag_calidad_incompleta"] = (
        df_calidad[columnas_requeridas_calidad]
        .isna()
        .any(axis = 1)
    )

    df_calidad["calidad_estandar_provisional"] = (
        df_calidad["calidad_radiante_publicada"]
        & ~df_calidad["flag_calidad_incompleta"]
        & df_calidad["fov_completo"]
    )

    df_calidad["calidad_estricta_provisional"] = (
        df_calidad["calidad_estandar_provisional"]
        & df_calidad["Num (stat)"].ge(4)
    )

    return df_calidad


## 4. Auditoría multianual


In [ ]:
# Cargar por separado los archivos de 2019 a 2025 y generar un resumen estructural por año

columnas_esenciales_auditoria = [
    "Unique trajectory (identifier)",
    "Beginning (UTC Time)",
    "IAU (code)",
    "Sol lon (deg)",
    "RAgeo (deg)",
    "RAgeo_sigma",
    "DECgeo (deg)",
    "DECgeo_sigma",
    "Vgeo (km/s)",
    "Vgeo_sigma",
    "Vavg (km/s)",
    "Vavg_sigma",
    "Peak (AbsMag)",
    "Qc (deg)",
    "MedianFitErr (arcsec)",
    "Num (stat)",
    "Beg in (FOV)",
    "End in (FOV)",
]

datos_por_anio = {}
registros_anuales = []

for anio in range(2019, 2026):
    nombre_archivo = f"traj_summary_monthly_{anio}12.txt"
    ruta_archivo = carpeta_raw / nombre_archivo
    
    print(f"Cargando diciembre de {anio}...")
    df_anio = cargar_archivo_gmn(ruta_archivo)
    
    datos_por_anio[anio] = df_anio
    
    identificadores_duplicados = df_anio["Unique trajectory (identifier)"].duplicated().sum()
    
    filas_con_faltantes_esenciales = df_anio[columnas_esenciales_auditoria].isna().any(axis = 1).sum()
    
    geminidas = df_anio["IAU (code)"].eq("GEM").sum()
    
    memoria_mb = df_anio.memory_usage(deep = True).sum() / 1024 ** 2
    
    registros_anuales.append(
        {
            "anio": anio,
            "filas": len(df_anio),
            "columnas": df_anio.shape[1],
            "nombres_unicos": (
                df_anio.columns.is_unique
            ),
            "ids_duplicados": (
                identificadores_duplicados
            ),
            "filas_con_faltantes_esenciales": (
                filas_con_faltantes_esenciales
            ),
            "geminidas_etiquetadas": (
                geminidas
            ),
            "fecha_inicial": (
                df_anio[
                    "Beginning (UTC Time)"
                ].min()
            ),
            "fecha_final": (
                df_anio[
                    "Beginning (UTC Time)"
                ].max()
            ),
            "memoria_mb": memoria_mb,
        }
    )
    
    print(f"Completado: {len(df_anio):,} filas.")
    
resumen_estructural_anual = pd.DataFrame(registros_anuales)

resumen_estructural_anual["memoria_mb"] = resumen_estructural_anual["memoria_mb"].round(2)

print("\nResumen estructural por año: ")
display(resumen_estructural_anual)


In [ ]:
# Comparar nombres y tipos de columnas entre años y localizar los faltantes esenciales

columnas_referencia = list(datos_por_anio[2019].columns)

tipos_referencia = datos_por_anio[2019].dtypes.astype(str)

registros_esquema = []
registros_faltantes = []

for anio, df_anio in datos_por_anio.items():
    columnas_anio = list(df_anio.columns)
    
    columnas_faltantes_esquema = sorted(set(columnas_referencia) - set(columnas_anio))
    
    columnas_adicionales_esquema = sorted(set(columnas_anio) - set(columnas_referencia))
    
    columnas_comunes = [columna for columna in columnas_referencia if columna in df_anio.columns]
    
    diferencias_tipos = []
    
    for columna in columnas_comunes:
        tipo_referencia = tipos_referencia[columna]
        tipo_anio = str(df_anio[columna].dtype)
        
        if tipo_anio != tipo_referencia:
            diferencias_tipos.append(f"{columna}: {tipo_referencia} —> {tipo_anio}")
            
    registros_esquema.append(
        {
            "anio": anio,
            "mismas_columnas": (
                len(
                    columnas_faltantes_esquema
                ) == 0
                and len(
                    columnas_adicionales_esquema
                ) == 0
            ),
            "mismo_orden": (
                columnas_anio
                == columnas_referencia
            ),
            "columnas_faltantes": (
                columnas_faltantes_esquema
            ),
            "columnas_adicionales": (
                columnas_adicionales_esquema
            ),
            "cantidad_diferencias_tipos": (
                len(diferencias_tipos)
            ),
            "diferencias_tipos": (
                diferencias_tipos
            ),
        }
    )
    
    for columna in columnas_esenciales_auditoria:
        mascara_faltante = df_anio[columna].isna()
        cantidad_faltante = mascara_faltante.sum()
        
        if cantidad_faltante > 0:
            faltantes_geminidas = (mascara_faltante & df_anio["IAU (code)"].eq("GEM")).sum()
            
            registros_faltantes.append(
                {
                    "anio": anio,
                    "columna": columna,
                    "faltantes": (
                        cantidad_faltante
                    ),
                    "porcentaje_anual": round(
                        (
                            100
                            * cantidad_faltante
                            / len(df_anio)
                        ),
                        4,
                    ),
                    "faltantes_en_geminidas": (
                        faltantes_geminidas
                    ),
                }
            )
            
resumen_compatibilidad = pd.DataFrame(registros_esquema)

detalle_faltantes_esenciales = pd.DataFrame(registros_faltantes)

print("Compatibilidad estructural respecto de 2019: ")
display(resumen_compatibilidad)

print("\nFaltantes por variable y año: ")

if detalle_faltantes_esenciales.empty: 
    print("No se encontraron faltantes en las variables esenciales.")
else:
    display(detalle_faltantes_esenciales)


In [ ]:
# Aplicar la preparación de calidad a todos los años y comprobar la exclusión de registros incompletos

datos_preparados_por_anio = {}
registros_control_calidad = []

for anio, df_anio in datos_por_anio.items():
    df_preparado = preparar_calidad_gmn(df_anio)
    datos_preparados_por_anio[anio] = df_preparado

    calidad_incompleta = df_preparado["flag_calidad_incompleta"]
    muestra_estandar = df_preparado["calidad_estandar_provisional"]
    muestra_estricta = df_preparado["calidad_estricta_provisional"]

    incompletas_en_estandar = (calidad_incompleta & muestra_estandar).sum()
    estrictas_fuera_de_estandar = (muestra_estricta & ~muestra_estandar).sum()

    if incompletas_en_estandar != 0:
        raise ValueError(
            f"En {anio} hay trayectorias con calidad incompleta dentro de la muestra estándar."
        )

    if estrictas_fuera_de_estandar != 0:
        raise ValueError(
            f"En {anio} hay trayectorias estrictas que no pertenecen a la muestra estándar."
        )

    registros_control_calidad.append(
        {
            "anio": anio,
            "calidad_incompleta": int(calidad_incompleta.sum()),
            "geminidas_con_calidad_incompleta": int(
                (
                    calidad_incompleta
                    & df_preparado["IAU (code)"].eq("GEM")
                ).sum()
            ),
            "incompletas_en_muestra_estandar": int(incompletas_en_estandar),
            "estrictas_fuera_de_estandar": int(estrictas_fuera_de_estandar),
            "filas_conservadas": len(df_preparado),
        }
    )

resumen_control_calidad = pd.DataFrame(registros_control_calidad)

display(resumen_control_calidad)


## 5. Integración de los siete años


In [ ]:
# Concatenar los siete años preparados y auditar el resultado de la integración

dataframes_preparados = [
    datos_preparados_por_anio[anio]
    for anio in range(2019, 2026)
]

df_integrado = pd.concat(
    dataframes_preparados,
    ignore_index = True,
    sort = False,
)

filas_esperadas = sum(
    len(df_anio)
    for df_anio in datos_por_anio.values()
)

if len(df_integrado) != filas_esperadas:
    raise ValueError(
        "El número de filas después de concatenar no coincide con la suma de los archivos individuales."
    )

if not df_integrado.columns.is_unique:
    raise ValueError(
        "La base integrada contiene nombres de columnas repetidos."
    )

ids_duplicados_globales = (
    df_integrado["Unique trajectory (identifier)"]
    .duplicated()
    .sum()
)

registros_integracion = []

for anio, df_anio in df_integrado.groupby("source_year", sort = True):
    es_geminida = df_anio["IAU (code)"].eq("GEM")
    cantidad_geminidas = es_geminida.sum()

    estandar = df_anio["calidad_estandar_provisional"]
    estricta = df_anio["calidad_estricta_provisional"]

    geminidas_estandar = (es_geminida & estandar).sum()
    geminidas_estrictas = (es_geminida & estricta).sum()

    filas_originales = len(datos_por_anio[anio])

    registros_integracion.append(
        {
            "anio": anio,
            "filas_originales": filas_originales,
            "filas_integradas": len(df_anio),
            "filas_coinciden": filas_originales == len(df_anio),
            "calidad_incompleta": int(
                df_anio["flag_calidad_incompleta"].sum()
            ),
            "geminidas_totales": int(cantidad_geminidas),
            "geminidas_estandar": int(geminidas_estandar),
            "retencion_estandar_gem_porcentaje": (
                round(
                    100 * geminidas_estandar / cantidad_geminidas,
                    2,
                )
                if cantidad_geminidas > 0
                else pd.NA
            ),
            "geminidas_estrictas": int(geminidas_estrictas),
            "retencion_estricta_gem_porcentaje": (
                round(
                    100 * geminidas_estrictas / cantidad_geminidas,
                    2,
                )
                if cantidad_geminidas > 0
                else pd.NA
            ),
        }
    )

resumen_integracion = pd.DataFrame(registros_integracion)

memoria_integrada_mb = (
    df_integrado.memory_usage(deep = True).sum()
    / 1024 ** 2
)

print("\nFilas esperadas:", filas_esperadas)
print("Filas integradas:", len(df_integrado))
print("Columnas integradas:", df_integrado.shape[1])
print("Nombres de columnas únicos:", df_integrado.columns.is_unique)
print("Identificadores duplicados globales:", ids_duplicados_globales)
print("Memoria de la base integrada:", round(memoria_integrada_mb, 2), "MB")
print("\nResumen de integración por año:")

display(resumen_integracion)


## 6. Exportación de la base integrada


In [ ]:
# Guardar la base integrada como Parquet y verificar el archivo generado

carpeta_interim = raiz_proyecto / "data" / "interim"
carpeta_tablas = raiz_proyecto / "reports" / "tables"

carpeta_interim.mkdir(parents = True, exist_ok = True)
carpeta_tablas.mkdir(parents = True, exist_ok = True)

ruta_parquet = carpeta_interim / "gmn_diciembre_2019_2025.parquet"
ruta_parquet_temporal = carpeta_interim / "gmn_diciembre_2019_2025.parquet.part"

ruta_resumen_csv = carpeta_tablas / "resumen_integracion_2019_2025.csv"

print("Guardando la base integrada...")

df_integrado.to_parquet(
    ruta_parquet_temporal,
    engine = "pyarrow",
    compression = "zstd",
    index = False,
)

ruta_parquet_temporal.replace(ruta_parquet)

resumen_integracion.to_csv(
    ruta_resumen_csv,
    index = False,
    encoding = "utf-8",
)

archivo_parquet = pq.ParquetFile(ruta_parquet)

filas_guardadas = archivo_parquet.metadata.num_rows
columnas_guardadas = archivo_parquet.metadata.num_columns
tamano_parquet_mb = ruta_parquet.stat().st_size / 1024 ** 2

if filas_guardadas != len(df_integrado):
    raise ValueError(
        "El archivo Parquet no contiene el número esperado de filas."
    )

if columnas_guardadas != df_integrado.shape[1]:
    raise ValueError(
        "El archivo Parquet no contiene el número esperado de columnas."
    )

print("Guardado completado correctamente.")
print("\nRuta del Parquet:", ruta_parquet)
print("Filas guardadas:", filas_guardadas)
print("Columnas guardadas:", columnas_guardadas)
print("Tamaño del Parquet:", round(tamano_parquet_mb, 2), "MB")
print("\nRuta del resumen CSV:", ruta_resumen_csv)
print("Filas del resumen:", len(resumen_integracion))
